<a href="https://colab.research.google.com/github/AarohiAnalyzes/rag-30-days/blob/main/Day_03_Mini_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mini reusable RAG system**

---
**1. Create Knowledge Base**

---



In [1]:
documents = [
    "Python is a high-level programming language widely used for web development, automation, data science, and artificial intelligence.",

    "Machine learning is a branch of artificial intelligence where computers learn patterns from data and use those patterns to make predictions.",

    "Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation to provide an LLM with relevant external context.",

    "Docker is a platform that packages applications and their dependencies into lightweight containers that can run consistently across different environments.",

    "An API is a set of rules that allows different software applications to communicate with each other.",

    "Redis is an in-memory data store commonly used for caching, session management, and fast data access.",

    "RabbitMQ is a message broker that allows applications to communicate asynchronously by sending and receiving messages through queues.",

    "Vector databases are specialized databases designed to store and search vector embeddings efficiently."
]

In [2]:
# check the length of documents
print(len(documents))

8


---
**2. Create Document Embeddings**

---

In [3]:
# load pretrained embedding model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
document_embeddings = model.encode(documents)
print(document_embeddings.shape)

(8, 384)


---
**3. Build the retrieve function**
*   encode the query
*   calculate the similarity score and retrun the query embedding


---

In [5]:
# import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
def retrieve(query):
  query_embedding = model.encode(query)

  similarity_scores = cosine_similarity(
      [query_embedding],
      document_embeddings
  )

  return similarity_scores

---
**4. Test the query**

---

In [7]:
query = "What is RAG?"

similarity_scores = retrieve(query)

print(similarity_scores)

[[0.05778092 0.00832052 0.40525043 0.102208   0.10217013 0.07323942
  0.07200249 0.0283327 ]]


---
**5. Rank the documents and find top-3 matching documents based on the similarity scores**

---

In [8]:
import numpy as np

top_3_indices = np.argsort(similarity_scores[0])[-3:][::-1]
print(top_3_indices)

[2 3 4]


---
**6. Find the documents related to the top 3 best index**

---


In [9]:
for i in top_3_indices:
  print(i, documents[i])

2 Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation to provide an LLM with relevant external context.
3 Docker is a platform that packages applications and their dependencies into lightweight containers that can run consistently across different environments.
4 An API is a set of rules that allows different software applications to communicate with each other.


---
**7. Create Context**

---

In [10]:
context = ""

for i in top_3_indices:
  context += documents[i] + "\n"

print(context)


Retrieval-Augmented Generation, or RAG, combines information retrieval with language generation to provide an LLM with relevant external context.
Docker is a platform that packages applications and their dependencies into lightweight containers that can run consistently across different environments.
An API is a set of rules that allows different software applications to communicate with each other.



---
8. Send the context to gemini

---

In [11]:
from google import genai

from google.colab import userdata
API_Key = userdata.get("Gemini_API_Key")
print("Key loaded:", API_Key is not None)
print("Key length:", len(API_Key) if API_Key else 0)

client = genai.Client(api_key=API_Key)

Key loaded: True
Key length: 53


In [12]:
query = "What is RAG?"

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

In [13]:
response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt
)

print(response.text)

Based on the provided context, RAG stands for Retrieval-Augmented Generation. It combines information retrieval with language generation to provide a large language model (LLM) with relevant external context.
